# IAPH - TP3 Chatbots basados en recuperación de la información

En inglés information retrieval chatbots

## Motor de búsqueda

* Búsqueda por palabras clave: Extrae palabras clave de la pregunta del usuario y busca coincidencias en las preguntas almacenadas.

* Similitud del coseno: Si has representado las preguntas como vectores (por ejemplo, usando TF-IDF o word embeddings), puedes usar la similitud del coseno para medir la distancia entre las preguntas.

* Embeddings: Utiliza modelos de word embeddings como por ejemplo Word2Vec para obtener representaciones semánticas de las preguntas y las consultas del usuario.

## Librerías

Debes trabajar en Python. Puedes usar las librerías sklearn, pandas, spacy o nltk o gensim para el punto de usar o buscar embeddings.

### 1)&nbsp;Instalo las librerías necesarias para trabajar.

In [2]:
# Instalo las librerías necesarias para trabajar.
!pip install spacy --progress-bar on --quiet
!python -m spacy download es_core_news_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 50.1 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


### 2)&nbsp;Importo las librerías necesarias para trabajar.

In [3]:
# Importo las librerías necesarias para trabajar.
import ipywidgets as widgets
import pandas as pd
import random
import spacy

from google.colab import data_table
from IPython.display import HTML, display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Actividades

### 1)&nbsp;Elaborar un dataset de preguntas y respuestas para crear un Chatbot para un aplicación particular. (3 puntos)

#### 1.1 Debe definir la aplicación (atención al cliente bancario, atención a estudiantes universitarios, etc).

> Aplicación elegida: **Chatbot de consulta para estudiantes de secundaria**.
>
> El chatbot responderá preguntas sobre contenidos generales de distintas áreas, como ciencias naturales, matemática, historia, geografía y literatura, utilizando una base de preguntas y respuestas en español.
>
> Uso un dataset que encontré en Hugging Face: https://huggingface.co/datasets/Dontallo/QASpanis

#### 1.2 El listado de preguntas y respuestas debe tener como mínimo 20 elementos pregunta - respuesta.

In [4]:
# Defino la URL del dataset.
url_dataset = "https://huggingface.co/datasets/Dontallo/QASpanish/resolve/main/dataset.csv"

# Cargo el dataset en un DataFrame.
dataframe = pd.read_csv(url_dataset)

# Muestro el dataset.
dataframe

,pregunta,respuesta
0,¿Qué es el ciclo del agua?,"El ciclo del agua, también conocido como ciclo..."
1,¿Cómo se forman las montañas?,Las montañas se forman principalmente por la t...
2,¿Qué es la fotosíntesis?,La fotosíntesis es el proceso por el cual las ...
3,¿Qué son los ecosistemas?,Un ecosistema es una comunidad de organismos v...
4,¿Qué es el sistema solar?,El sistema solar es un conjunto de objetos cel...
...,...,...
99,¿Cuál fue el primer presidente de Bolivia?,El primer presidente constitucional de Bolivia...
100,¿Cuántos lados tiene un cuadrado?,Un cuadrado es una figura geométrica plana que...
101,¿Qué son los productores en una cadena aliment...,"En una cadena alimentaria, los productores son..."
102,¿Qué evento desencadenó la Primera Guerra Mund...,El evento que desencadenó la Primera Guerra Mu...


In [5]:
# ¿Tengo preguntas repetidas? Busco por la palabra fotosíntesis.
preguntas_fotosintesis = dataframe[
    dataframe["pregunta"].str.contains("fotosíntesis", case=False, na=False)
]

# ¿Tengo preguntas repetidas? Muestro las preguntas encontradas.
preguntas_fotosintesis

,pregunta,respuesta
2,¿Qué es la fotosíntesis?,La fotosíntesis es el proceso por el cual las ...
34,¿Qué es la fotosíntesis?,La fotosíntesis es el proceso mediante el cual...
43,¿Qué es la fotosíntesis?,La fotosíntesis es el proceso mediante el cual...
49,¿Qué es la fotosíntesis?,La fotosíntesis es el proceso mediante el cual...
57,Descríbeme la fotosíntesis.,La fotosíntesis es el proceso mediante el cual...
88,Explícame la fotosíntesis.,La fotosíntesis es el proceso mediante el cual...


In [6]:
# Elimino las preguntas repetidas y conservo la primera aparición.
dataframe_limpio = dataframe.drop_duplicates(subset="pregunta", keep="first").reset_index(drop=True)

# Muestro el DataFrame actualizado.
dataframe_limpio

,pregunta,respuesta
0,¿Qué es el ciclo del agua?,"El ciclo del agua, también conocido como ciclo..."
1,¿Cómo se forman las montañas?,Las montañas se forman principalmente por la t...
2,¿Qué es la fotosíntesis?,La fotosíntesis es el proceso por el cual las ...
3,¿Qué son los ecosistemas?,Un ecosistema es una comunidad de organismos v...
4,¿Qué es el sistema solar?,El sistema solar es un conjunto de objetos cel...
...,...,...
91,¿Cuál fue el primer presidente de Bolivia?,El primer presidente constitucional de Bolivia...
92,¿Cuántos lados tiene un cuadrado?,Un cuadrado es una figura geométrica plana que...
93,¿Qué son los productores en una cadena aliment...,"En una cadena alimentaria, los productores son..."
94,¿Qué evento desencadenó la Primera Guerra Mund...,El evento que desencadenó la Primera Guerra Mu...


###  2)&nbsp;Crear el chatbot utilizando TF-IDF y similitud del coseno. (1 punto)

#### 2.1 Vectorizo las preguntas utilizando TF-IDF.

In [7]:
# Separo las preguntas y las respuestas del dataset.
preguntas = dataframe_limpio["pregunta"].tolist()
respuestas = dataframe_limpio["respuesta"].tolist()

# Vectorizo las preguntas utilizando TF-IDF.
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(preguntas)

# Muestro la matriz generada.
tfidf_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 481 stored elements and shape (96, 140)>

> **tfidf_matrix** guarda la representación numérica de todas las preguntas para poder compararlas con una consulta nueva.

#### 2.2 Defino una función para encontrar la respuesta con mayor similitud.

In [8]:
# Defino una función para encontrar la respuesta con mayor similitud.
def responder_pregunta(consulta_usuario):
    # Convierto la consulta del usuario en un vector TF-IDF.
    consulta_vec = vectorizer.transform([consulta_usuario])

    # Comparo la consulta con todas las preguntas del dataset.
    similitudes = cosine_similarity(consulta_vec, tfidf_matrix).flatten()

    # Obtengo el índice de la pregunta con mayor similitud.
    indice_mejor_coincidencia = similitudes.argmax()

    # Devuelvo la respuesta correspondiente.
    return respuestas[indice_mejor_coincidencia]

> **argmax()** devuelve la posición del valor más alto de la lista de similitudes, es decir, identifica cuál es la pregunta del dataset que más se parece a la consulta del usuario.

#### 2.3 Pruebo una pregunta/respuesta rápida.

In [9]:
# Defino una consulta de prueba.
consulta_usuario = "¿Qué son los ecosistemas?"

# Obtengo la respuesta del chatbot.
respuesta = responder_pregunta(consulta_usuario)

# Muestro la respuesta obtenida.
print(respuesta)

Un ecosistema es una comunidad de organismos vivos que interactúan entre sí y con su entorno físico en un área específica. Los ecosistemas incluyen componentes bióticos (organismos vivos, como plantas, animales y microorganismos) y abióticos (factores no vivos, como el clima, el suelo, el agua y los nutrientes). Los ecosistemas pueden variar en tamaño, desde una pequeña charca hasta un gran bosque o un océano. Las interacciones dentro de un ecosistema incluyen procesos como la depredación, la competencia, la simbiosis y el ciclo de nutrientes. Los ecosistemas son dinámicos y pueden cambiar con el tiempo debido a factores naturales y humanos.


#### 2.4 Chatbot TF-IDF.

In [10]:
# Creo la caja de texto para escribir las preguntas.
entrada = widgets.Text(
    description="Pregunta:",
    layout=widgets.Layout(width="90%")
)

# Selecciono preguntas aleatorias del dataset como sugerencias.
preguntas_sugeridas = dataframe_limpio["pregunta"].sample(5).tolist()

# Construyo el texto con las preguntas de ejemplo en una sola línea.
texto_ejemplos = ", ".join(f'"{pregunta}"' for pregunta in preguntas_sugeridas)

# Creo una leyenda con preguntas de ejemplo.
sugerencias = HTML(
    f'<div style="width: 90%; color: #777; font-size: 12px; margin-top: 5px; padding-left: 89px;">'
    f'Ej: {texto_ejemplos}.'
    f'</div>'
)

# Defino la interacción con el chatbot.
def responder_chat(widget):
    consulta_usuario = widget.value

    # Obtengo la respuesta del chatbot.
    respuesta = responder_pregunta(consulta_usuario)

    # Muestro la pregunta y su respuesta.
    display(HTML(
        f'<div style="width: 90%; color: #333; font-size: 13px; margin-top: 15px; padding-left: 0;">'
        f'<b>Pregunta:</b> {consulta_usuario}<br>'
        f'<b>Respuesta:</b> {respuesta}'
        f'</div>'
    ))

    # Limpio la caja de texto.
    widget.value = ""

# Envío la pregunta al presionar Enter.
entrada.on_submit(responder_chat)

# Muestro la caja de texto y las sugerencias.
display(entrada, sugerencias)

Text(value='', description='Pregunta:', layout=Layout(width='90%'))

### 3)&nbsp;Crear otro chatbot utilizando embeddings. Indique cuál embedding (1 punto) pre-entrenado de las librerías Spacy o nltk o gensim eligió. Justifique.

> Embedding elegido: **modelo preentrenado `es_core_news_md` de spaCy**.
>
> Elijo este modelo porque trabaja con texto en español y me permite obtener representaciones vectoriales de las preguntas para comparar su similitud semántica. Considero que resulta adecuado para el dataset que utilizo, ya que contiene preguntas y respuestas en español.

#### 3.1 Vectorizo las preguntas utilizando embeddings.

In [11]:
# Cargo el modelo preentrenado en español.
modelo_embeddings = spacy.load("es_core_news_md")

# Obtengo los vectores de todas las preguntas del dataset.
vectores_preguntas = [
    modelo_embeddings(pregunta).vector
    for pregunta in preguntas
]

# Muestro la cantidad de vectores generados.
len(vectores_preguntas)

96

#### 3.2 Defino una función para encontrar la respuesta con mayor similitud.

In [12]:
# Defino una función para encontrar la respuesta con mayor similitud semántica.
def responder_pregunta_embeddings(consulta_usuario):
    # Obtengo el vector de la consulta del usuario.
    vector_consulta = modelo_embeddings(consulta_usuario).vector

    # Comparo la consulta con todos los vectores de las preguntas.
    similitudes = cosine_similarity([vector_consulta],vectores_preguntas).flatten()

    # Obtengo el índice de la pregunta con mayor similitud.
    indice_mejor_coincidencia = similitudes.argmax()

    # Devuelvo la respuesta correspondiente.
    return respuestas[indice_mejor_coincidencia]

#### 3.3 Chatbot embeddings.

In [13]:
# Creo la caja de texto para escribir las preguntas.
entrada = widgets.Text(
    description="Pregunta:",
    layout=widgets.Layout(width="90%")
)

# Selecciono preguntas aleatorias del dataset como sugerencias.
preguntas_sugeridas = dataframe_limpio["pregunta"].sample(5).tolist()

# Construyo el texto con las preguntas de ejemplo en una sola línea.
texto_ejemplos = ", ".join(f'"{pregunta}"' for pregunta in preguntas_sugeridas)

# Creo una leyenda con preguntas de ejemplo.
sugerencias = HTML(
    f'<div style="width: 90%; color: #777; font-size: 12px; margin-top: 5px; padding-left: 89px;">'
    f'Ej: {texto_ejemplos}.'
    f'</div>'
)

# Defino la interacción con el chatbot.
def responder_chat_embeddings(widget):
    consulta_usuario = widget.value

    # Obtengo la respuesta del chatbot.
    respuesta = responder_pregunta_embeddings(consulta_usuario)

    # Muestro la pregunta y su respuesta.
    display(HTML(
        f'<div style="width: 90%; color: #333; font-size: 13px; margin-top: 15px; padding-left: 0;">'
        f'<b>Pregunta:</b> {consulta_usuario}<br>'
        f'<b>Respuesta:</b> {respuesta}'
        f'</div>'
    ))

    # Limpio la caja de texto.
    widget.value = ""

# Envío la pregunta al presionar Enter.
entrada.on_submit(responder_chat_embeddings)

# Muestro la caja de texto y las sugerencias.
display(entrada, sugerencias)

Text(value='', description='Pregunta:', layout=Layout(width='90%'))

### 4)&nbsp;Muestra ambos chatbots funcionando (1 punto)

Adjuntar la lista de preguntas y respuestas utilizadas para probar el funcionamiento.

Releve o indique cuáles respondió correctamente y cuáles no.

#### 4.1 Tomo algunas preguntas al azar y las modifico para que no sean exactas.

In [14]:
# Defino una función para modificar las preguntas originales.
def modificar_pregunta(pregunta):
    modificaciones = {
        "¿Qué es ": ["Explícame qué es ", "¿En qué consiste ", "¿Cómo definirías "],
        "¿Qué son ": ["Explícame qué son ", "¿Cómo definirías ", "¿A qué se llama "],
        "¿Cuál es ": ["Decime cuál es ", "Quiero saber cuál es ", "¿Podés decirme cuál es "],
        "¿Cuáles son ": ["Decime cuáles son ", "Quiero saber cuáles son ", "¿Podés decirme cuáles son "],
        "¿Cómo ": ["¿De qué manera ", "Explícame cómo ", "Quiero saber cómo "],
        "¿Por qué ": ["Explícame por qué ", "Quiero saber por qué ", "¿Cuál es la razón por la que "],
        "¿Quién fue ": ["Contame quién fue ", "Quiero saber quién fue ", "¿Podés explicarme quién fue "]
    }

    # Busco una forma alternativa de formular la pregunta.
    for inicio, alternativas in modificaciones.items():
        if pregunta.startswith(inicio):
            return random.choice(alternativas) + pregunta[len(inicio):]

    # Si no encuentro un patrón, modifico la forma general de la pregunta.
    return "Quiero saber: " + pregunta.replace("¿", "").replace("?", "")


# Selecciono quince preguntas aleatorias del dataset.
muestra = dataframe_limpio.sample(15).copy()

# Convierto el índice original en una columna identificadora.
muestra = muestra.reset_index().rename(
    columns={
        "index": "id",
        "pregunta": "pregunta_original"
    }
)

# Genero una versión modificada de cada pregunta.
muestra["pregunta_modificada"] = muestra["pregunta_original"].apply(modificar_pregunta)

# Muestro las preguntas seleccionadas.
muestra[["id", "pregunta_original", "pregunta_modificada"]]

,id,pregunta_original,pregunta_modificada
0,83,Explícame las leyes de Newton.,Quiero saber: Explícame las leyes de Newton.
1,82,¿Qué fue la Revolución Francesa?,Quiero saber: Qué fue la Revolución Francesa
2,63,Descríbeme el sistema solar.,Quiero saber: Descríbeme el sistema solar.
3,9,¿Qué es la materia?,Explícame qué es la materia?
4,4,¿Qué es el sistema solar?,¿Cómo definirías el sistema solar?
5,49,Explícame el sistema solar.,Quiero saber: Explícame el sistema solar.
6,15,¿Qué es la electricidad?,¿Cómo definirías la electricidad?
7,95,¿Qué cultura antigua construyó Machu Picchu?,Quiero saber: Qué cultura antigua construyó Ma...
8,23,¿Qué es la reproducción?,¿En qué consiste la reproducción?
9,21,¿Qué es la célula?,¿En qué consiste la célula?


#### 4.2 Le pregunto a los dos tipos de chatbots y comparo los resultados.

In [15]:
# Obtengo las respuestas de ambos chatbots para cada pregunta modificada.
muestra["respuesta_tfidf"] = muestra["pregunta_modificada"].apply(responder_pregunta)
muestra["respuesta_embeddings"] = muestra["pregunta_modificada"].apply(responder_pregunta_embeddings)

# Indico si cada chatbot devolvió exactamente la respuesta esperada.
muestra["acierto_tfidf"] = muestra["respuesta_tfidf"] == muestra["respuesta"]
muestra["acierto_embeddings"] = muestra["respuesta_embeddings"] == muestra["respuesta"]

# Creo una tabla con los resultados.
tabla_resultados = muestra[[
    "id",
    "pregunta_original",
    "pregunta_modificada",
    "respuesta",
    "respuesta_tfidf",
    "acierto_tfidf",
    "respuesta_embeddings",
    "acierto_embeddings"
]].copy()

# Agrego una fila final con el total de aciertos.
tabla_resultados.loc["TOTAL ACIERTOS"] = [
    "",
    "",
    "",
    "",
    "",
    tabla_resultados["acierto_tfidf"].sum(),
    "",
    tabla_resultados["acierto_embeddings"].sum()
]

# Muestro la tabla comparativa.
tabla_resultados

,id,pregunta_original,pregunta_modificada,respuesta,respuesta_tfidf,acierto_tfidf,respuesta_embeddings,acierto_embeddings
0,83,Explícame las leyes de Newton.,Quiero saber: Explícame las leyes de Newton.,"Las leyes de Newton, formuladas por Sir Isaac ...","Las leyes de Newton, formuladas por Sir Isaac ...",1,"Las leyes de Newton, formuladas por Sir Isaac ...",1
1,82,¿Qué fue la Revolución Francesa?,Quiero saber: Qué fue la Revolución Francesa,La Revolución Francesa fue un período de radic...,La Revolución Francesa fue un período de radic...,1,La Revolución Francesa fue un período de radic...,1
2,63,Descríbeme el sistema solar.,Quiero saber: Descríbeme el sistema solar.,El sistema solar es un conjunto de cuerpos cel...,El sistema solar es un conjunto de cuerpos cel...,1,El sistema solar es un conjunto de cuerpos cel...,1
3,9,¿Qué es la materia?,Explícame qué es la materia?,La materia es todo aquello que tiene masa y oc...,La materia es todo aquello que tiene masa y oc...,1,La materia es todo aquello que tiene masa y oc...,1
4,4,¿Qué es el sistema solar?,¿Cómo definirías el sistema solar?,El sistema solar es un conjunto de objetos cel...,El sistema solar es un conjunto de objetos cel...,1,El corazón es un órgano muscular que bombea sa...,0
5,49,Explícame el sistema solar.,Quiero saber: Explícame el sistema solar.,El sistema solar es un conjunto de cuerpos cel...,El sistema solar es un conjunto de cuerpos cel...,1,El sistema solar es un conjunto de cuerpos cel...,1
6,15,¿Qué es la electricidad?,¿Cómo definirías la electricidad?,La electricidad es una forma de energía que se...,La electricidad es una forma de energía que se...,1,La electricidad es una forma de energía que se...,1
7,95,¿Qué cultura antigua construyó Machu Picchu?,Quiero saber: Qué cultura antigua construyó Ma...,Machu Picchu fue construida por la civilizació...,Machu Picchu fue construida por la civilizació...,1,Machu Picchu fue construida por la civilizació...,1
8,23,¿Qué es la reproducción?,¿En qué consiste la reproducción?,La reproducción es el proceso mediante el cual...,La reproducción es el proceso mediante el cual...,1,La tectónica de placas es la teoría que explic...,0
9,21,¿Qué es la célula?,¿En qué consiste la célula?,La célula es la unidad básica de la vida. Todo...,La célula es la unidad básica de la vida. Todo...,1,La célula es la unidad básica de la vida. Todo...,1


#### 4.3 Tabla interactiva para ver preguntas y respuestas, para entender los resultados.

In [16]:
# Creo una copia del DataFrame limpio.
tabla_dataset = dataframe_limpio.copy()

# Creo los filtros.
busqueda_pregunta = widgets.Text(
    placeholder="Buscar en preguntas...",
    layout=widgets.Layout(width="100%")
)

busqueda_respuesta = widgets.Text(
    placeholder="Buscar en respuestas...",
    layout=widgets.Layout(width="100%")
)

# Creo los botones para limpiar los filtros.
limpiar_pregunta = widgets.Button(
    description="×",
    layout=widgets.Layout(width="35px")
)

limpiar_respuesta = widgets.Button(
    description="×",
    layout=widgets.Layout(width="35px")
)

# Creo el espacio para mostrar la tabla.
salida_busqueda = widgets.Output()

# Defino la búsqueda.
def buscar_dataset(change=None):
    resultados = tabla_dataset.copy()

    if busqueda_pregunta.value:
        resultados = resultados[
            resultados["pregunta"].str.contains(
                busqueda_pregunta.value,
                case=False,
                na=False
            )
        ]

    if busqueda_respuesta.value:
        resultados = resultados[
            resultados["respuesta"].str.contains(
                busqueda_respuesta.value,
                case=False,
                na=False
            )
        ]

    with salida_busqueda:
        salida_busqueda.clear_output(wait=True)

        display(
            resultados.head(10).style.set_table_attributes(
                'style="width:100%; table-layout:fixed;"'
            )
        )

# Actualizo la tabla cuando cambia un filtro.
busqueda_pregunta.observe(buscar_dataset, names="value")
busqueda_respuesta.observe(buscar_dataset, names="value")

# Limpio cada filtro al presionar la cruz.
limpiar_pregunta.on_click(lambda _: setattr(busqueda_pregunta, "value", ""))
limpiar_respuesta.on_click(lambda _: setattr(busqueda_respuesta, "value", ""))

# Muestro los filtros con espacio entre ambos grupos.
display(
    widgets.HBox(
        [
            widgets.HBox(
                [busqueda_pregunta, limpiar_pregunta],
                layout=widgets.Layout(width="49%")
            ),
            widgets.HBox(
                [busqueda_respuesta, limpiar_respuesta],
                layout=widgets.Layout(width="49%")
            )
        ],
        layout=widgets.Layout(
            width="100%",
            justify_content="space-between"
        )
    ),
    salida_busqueda
)

# Muestro inicialmente los primeros diez registros.
buscar_dataset()

Output()

### 5)&nbsp;Añade tus conclusiones de todo lo realizado (2 punto)

* Resalte e indique en cuáles respuestas falla o tiene problemas.
* Cuáles preguntas confunde.
* Compare los resultados de los chatbots.

> En esta prueba, TF-IDF obtuvo más aciertos que el modelo de embeddings. Considero que esto ocurre porque las preguntas modificadas conservan muchas veces el concepto principal de la pregunta original. En TF-IDF, esas palabras específicas tienen mucho peso y permiten diferenciar con facilidad una pregunta de otra.
>
> Por ejemplo, términos como:
>
> - **biodiversidad**
> - **continentes**
> - **clima**
> - **magnetismo**
>
> son muy útiles para identificar el tema correcto, aunque cambie la forma de hacer la pregunta.
>
> En cambio, el modelo de embeddings representa la frase completa como un vector. Esto puede hacer que dos preguntas con una estructura parecida queden demasiado próximas aunque el concepto principal sea distinto. Por ejemplo:
>
> - **“¿Qué son los continentes?”** y **“¿Qué es el magnetismo?”** tienen una estructura muy similar, pero las palabras importantes, **continentes** y **magnetismo**, pertenecen a temas completamente diferentes.
>
> También hubo errores menos explicables por la estructura, por ejemplo:
>
> - **“¿En qué consiste la biodiversidad?”** → devolvió una respuesta sobre la Primera Guerra Mundial.
> - **“¿En qué consiste el clima?”** → devolvió una respuesta sobre el corazón.
>
> Esto muestra que, en algunas consultas, el embedding no da suficiente peso al concepto central y termina considerando más cercana otra pregunta que no corresponde. En este dataset, como las preguntas son cortas y contienen términos muy específicos, TF-IDF funciona mejor porque esas palabras clave resultan muy discriminantes.

## No olvides

* Explicar tus decisiones y configuraciones. Añadir tus conclusiones.
* Anunciar en el foro cuál será tu aplicación y postear tu entrega y tus avances.
* Debes subir tu notebook a un repo GitHub público de tu propiedad compartido + enlace colab.
* Documentar todo el proceso.
* Citar tus fuentes.

> En este trabajo comparé dos chatbots de recuperación: uno basado en **TF-IDF** y otro en **embeddings con `es_core_news_md` de spaCy**.
>
> Probé ambos chatbots de forma manual, mediante cajas de texto en las que se pueden escribir distintas consultas y observar las respuestas obtenidas.
>
> Además, realicé una prueba automática en la que seleccioné preguntas al azar del dataset, modifiqué su redacción y comparé las respuestas de ambos modelos.
>
> En las pruebas realizadas, **TF-IDF obtuvo mejores resultados**, especialmente porque muchas preguntas contienen términos específicos que permiten identificar bien el tema. El modelo de embeddings presentó más confusiones entre preguntas distintas, lo que muestra que un método más complejo no necesariamente funciona mejor y que el resultado también depende de las características del dataset.
>
> **Fuentes utilizadas:**
> - Notebooks de clase.
> - Búsquedas en Internet.
> - Dataset de Hugging Face.
> - Documentación de spaCy y scikit-learn.
> - ChatGPT y Gemini para esclarecer dudas.

#
---

#IAPH - TP5 Chatbot RAG y evaluación de embeddings

## 1)&nbsp;CONSIGNAS a copiar y pegar en su notebook para luego RESOLVER

### a) Creación del conjunto de datos de evaluación.

Además del dataset original que creó, debe crear un dataset de prueba o evaluación con la misma lógica: preguntas y respuestas.

#### I. Instalo las librerías necesarias para trabajar.

In [29]:
# Instalo las librerías necesarias para el chatbot RAG.
!pip install transformers sentence-transformers accelerate faiss-cpu --quiet

#### II. Agrego las librerías necesarias para trabajar.

In [30]:
# Importo las librerías necesarias para trabajar.
import faiss
import numpy as np
import re
import torch

from html import escape
from IPython.display import display, HTML

print("Importando Sentence Transformers...")
from sentence_transformers import SentenceTransformer

print("Importando Transformers...")
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Todas las librerías se importaron correctamente.")

Importando Sentence Transformers...
Importando Transformers...
Todas las librerías se importaron correctamente.


#### III. Adiciono preguntas para el entrenamiento.

In [31]:
# Guardo el conjunto de entrenamiento original.
dataframe_entrenamiento_original = dataframe_limpio.copy()

# Amplío el conjunto de entrenamiento con preguntas de ciencias naturales.
preguntas_adicionales = [
    ("¿Qué ocurre cuando la luz encuentra un objeto opaco?", "Un objeto opaco impide que la luz lo atraviese. Detrás del objeto se forma una región con menor iluminación porque los rayos luminosos quedan bloqueados."),
    ("¿Qué diferencia existe entre materiales transparentes y opacos?", "Los materiales transparentes permiten que la luz los atraviese y que se distingan objetos detrás de ellos. Los opacos bloquean su transmisión."),
    ("¿Qué es la reflexión de la luz?", "La reflexión ocurre cuando la luz incide sobre una superficie y cambia de dirección, regresando al medio del que provenía."),
    ("¿Qué es la refracción de la luz?", "La refracción es el cambio de dirección de la luz cuando pasa entre medios con diferentes índices de refracción, generalmente acompañado por un cambio de velocidad."),
    ("¿Cómo se propaga la luz en un medio homogéneo?", "La luz se propaga en línea recta cuando atraviesa un medio homogéneo y transparente, siempre que no existan obstáculos ni cambios en sus propiedades."),
    ("¿Qué es una fuente luminosa?", "Es un cuerpo que emite luz propia, como el Sol, una lámpara encendida o una llama."),
    ("¿Qué sucede cuando un material absorbe luz?", "La energía luminosa absorbida se transforma en otras formas de energía, frecuentemente calor, según las características del material."),
    ("¿De qué depende el tamaño de una sombra?", "Depende de la posición y el tamaño de la fuente luminosa, las dimensiones del objeto y las distancias entre la fuente, el objeto y la superficie iluminada."),
    ("¿Cómo absorben las plantas el agua del suelo?", "Las raíces incorporan agua del suelo, principalmente mediante sus pelos absorbentes. El agua se transporta hacia otras partes de la planta a través del xilema."),
    ("¿Cómo obtienen minerales las plantas?", "Las raíces absorben iones minerales disueltos en el agua del suelo. Estos nutrientes son necesarios para el crecimiento y el funcionamiento celular."),
    ("¿Qué permite que una planta permanezca sujeta al suelo?", "El sistema de raíces proporciona anclaje y estabilidad, ayudando a mantener la planta erguida frente al viento y otras fuerzas externas."),
    ("¿Cuál es la función principal del tallo?", "El tallo sostiene hojas, flores y frutos. Además, contiene tejidos que transportan agua, minerales y sustancias orgánicas entre los órganos de la planta."),
    ("¿Qué papel cumplen las hojas en las plantas?", "Las hojas realizan gran parte de la fotosíntesis y permiten el intercambio de gases con el ambiente mediante estructuras llamadas estomas."),
    ("¿Qué necesita una planta para fabricar sustancias orgánicas mediante fotosíntesis?", "Necesita energía luminosa, agua y dióxido de carbono. Mediante la fotosíntesis produce compuestos orgánicos y libera oxígeno."),
    ("¿Qué función cumple la clorofila?", "La clorofila es un pigmento que absorbe energía luminosa y participa en las reacciones fotosintéticas que permiten convertir esa energía en energía química."),
    ("¿Qué diferencia existe entre fotosíntesis y respiración celular?", "La fotosíntesis utiliza energía luminosa para sintetizar compuestos orgánicos. La respiración celular permite obtener energía utilizable a partir de compuestos orgánicos."),
    ("¿Qué caracteriza a un material conductor de electricidad?", "Un conductor permite que las cargas eléctricas se desplacen con relativa facilidad. Los metales suelen conducir bien porque contienen electrones móviles."),
    ("¿Qué caracteriza a un material aislante?", "Un aislante dificulta el movimiento de cargas eléctricas. El plástico, el vidrio y la goma suelen utilizarse como aislantes eléctricos."),
    ("¿Qué es la corriente eléctrica?", "La corriente eléctrica es el flujo de carga eléctrica a través de un material o circuito. Se mide en amperios."),
    ("¿Qué es la resistencia eléctrica?", "La resistencia eléctrica expresa la oposición de un material o componente al paso de la corriente eléctrica y se mide en ohmios."),
    ("¿Por qué se utilizan cables de cobre en instalaciones eléctricas?", "El cobre tiene buena conductividad eléctrica, puede transformarse en cables y ofrece propiedades mecánicas adecuadas para transportar corriente."),
    ("¿Qué función cumple el recubrimiento plástico de un cable?", "El recubrimiento plástico actúa como aislante y ayuda a evitar contactos eléctricos accidentales y cortocircuitos."),
    ("¿Qué ocurre en un circuito eléctrico abierto?", "Cuando un circuito está abierto, el recorrido conductor está interrumpido y no circula corriente a través de ese camino."),
    ("¿Cómo se relacionan el voltaje, la corriente y la resistencia?", "En un conductor óhmico, la ley de Ohm establece que la diferencia de potencial es igual al producto de la corriente por la resistencia."),
    ("¿Qué es una cadena alimentaria?", "Es una representación de la transferencia de materia y energía entre organismos mediante relaciones de alimentación."),
    ("¿Qué diferencia existe entre productores y consumidores?", "Los productores elaboran materia orgánica a partir de sustancias inorgánicas utilizando una fuente de energía. Los consumidores obtienen materia orgánica alimentándose de otros organismos."),
    ("¿Qué es la depredación?", "La depredación es una interacción ecológica en la que un organismo captura y consume a otro, transfiriendo materia y energía entre ambos."),
    ("¿Cómo puede la depredación afectar a una población?", "La depredación puede disminuir la cantidad de individuos de una población y contribuir a regular su tamaño junto con otros factores ambientales."),
    ("¿Qué ocurre cuando disminuye considerablemente una población de presas?", "Los depredadores especializados pueden encontrar menos alimento y disminuir su supervivencia o reproducción. También pueden modificar sus hábitos alimentarios."),
    ("¿Qué es una red trófica?", "Es el conjunto de cadenas alimentarias interconectadas de un ecosistema. Representa las diferentes relaciones de alimentación entre sus organismos."),
    ("¿Qué función cumplen los descomponedores?", "Los descomponedores degradan materia orgánica muerta y contribuyen al reciclaje de nutrientes que pueden volver a utilizar los productores."),
    ("¿Qué diferencia existe entre competencia y depredación?", "La competencia ocurre cuando organismos utilizan un recurso limitado. En la depredación, un organismo captura y consume a otro."),
    ("¿Cómo se origina el sonido?", "El sonido se origina mediante vibraciones que producen perturbaciones mecánicas y se transmiten a través de un medio material."),
    ("¿Por qué el sonido necesita un medio material?", "Porque las ondas sonoras transmiten energía mediante las vibraciones e interacciones de las partículas de un medio, como aire, agua o sólidos."),
    ("¿Puede propagarse el sonido en el vacío?", "No. El sonido necesita materia para transmitir vibraciones, por lo que no puede propagarse en el vacío."),
    ("¿Qué factores determinan la velocidad del sonido?", "La velocidad del sonido depende de las propiedades elásticas y de la densidad del medio, además de factores como la temperatura."),
    ("¿Por qué el sonido suele viajar más rápido en sólidos que en gases?", "En muchos sólidos, la elevada rigidez permite transmitir perturbaciones mecánicas más rápidamente que en los gases."),
    ("¿Qué representa la frecuencia de una onda sonora?", "La frecuencia indica la cantidad de oscilaciones por segundo y se relaciona con el tono percibido: frecuencias mayores producen sonidos más agudos."),
    ("¿Qué relación existe entre amplitud e intensidad sonora?", "La amplitud de la vibración se relaciona con la energía transportada por la onda. Una mayor amplitud suele producir mayor intensidad sonora."),
    ("¿Qué es el eco?", "El eco es la percepción de un sonido reflejado que regresa al oyente después de rebotar en una superficie suficientemente distante."),
    ("¿Qué es el sistema nervioso autónomo?", "Es la parte del sistema nervioso que regula funciones involuntarias como la frecuencia cardíaca, la digestión y otros procesos internos."),
    ("¿Qué función tiene el sistema nervioso simpático?", "El sistema simpático prepara al organismo para afrontar situaciones de alerta, aumentando la frecuencia cardíaca y modificando otras funciones corporales."),
    ("¿Qué es la respuesta de lucha o huida?", "Es una reacción fisiológica ante amenazas que prepara al organismo para responder rápidamente mediante cambios cardiovasculares, respiratorios y metabólicos."),
    ("¿Qué efectos produce la adrenalina en el organismo?", "La adrenalina puede aumentar la frecuencia cardíaca, favorecer el suministro de sangre a los músculos y preparar al cuerpo para situaciones de esfuerzo o peligro."),
    ("¿Qué es la frecuencia cardíaca?", "Es la cantidad de latidos del corazón por minuto. Puede variar según el ejercicio, las emociones, la temperatura y el estado de salud."),
    ("¿Qué función cumple el corazón?", "El corazón bombea sangre a través de los vasos sanguíneos y permite distribuir oxígeno y nutrientes, además de transportar sustancias de desecho."),
    ("¿Qué sucede con la circulación sanguínea durante el ejercicio?", "Durante el ejercicio suele aumentar el gasto cardíaco para transportar más oxígeno y nutrientes a los músculos activos."),
    ("¿Cómo se relacionan el sistema respiratorio y el circulatorio?", "El sistema respiratorio permite intercambiar oxígeno y dióxido de carbono en los pulmones. El sistema circulatorio transporta estos gases entre los pulmones y los tejidos."),
    ("¿Qué es la fuerza gravitatoria?", "Es la fuerza de atracción que existe entre los cuerpos con masa. Su intensidad depende de las masas y de la distancia entre ellos."),
    ("¿Qué establece la ley de gravitación universal?", "Establece que la fuerza gravitatoria entre dos cuerpos es proporcional al producto de sus masas e inversamente proporcional al cuadrado de la distancia entre sus centros."),
    ("¿Qué diferencia existe entre masa y peso?", "La masa representa una propiedad de la materia y se mide en kilogramos. El peso es una fuerza gravitatoria y se mide en newtons."),
    ("¿Qué es la aceleración?", "La aceleración es la variación de la velocidad por unidad de tiempo. Puede modificar la rapidez, la dirección o ambas."),
    ("¿Qué efecto produce una fuerza neta sobre un cuerpo?", "Una fuerza neta provoca una aceleración del cuerpo de acuerdo con la segunda ley de Newton."),
    ("¿Por qué caen los objetos cuando los soltamos cerca de la superficie terrestre?", "La atracción gravitatoria de la Tierra produce una aceleración hacia su centro, haciendo que los objetos caigan si no existe un soporte que lo impida."),
    ("¿Por qué los planetas permanecen en órbita alrededor del Sol?", "La atracción gravitatoria del Sol modifica continuamente la dirección del movimiento de los planetas, permitiendo sus trayectorias orbitales."),
    ("¿Cómo afecta la distancia a la atracción gravitatoria?", "Cuando aumenta la distancia entre dos cuerpos, la intensidad de su atracción gravitatoria disminuye según una relación inversa con el cuadrado de la distancia."),
    ("¿Qué es la biodiversidad?", "Es la variedad de formas de vida, incluyendo la diversidad genética, la diversidad de especies y la diversidad de ecosistemas."),
    ("¿Qué es la estabilidad ecológica?", "Es la capacidad de un ecosistema para mantener su funcionamiento frente a cambios o perturbaciones, dentro de determinados límites."),
    ("¿Qué es la resiliencia de un ecosistema?", "Es su capacidad para recuperarse o reorganizarse después de una perturbación y conservar funciones ecológicas fundamentales."),
    ("¿Por qué son importantes los polinizadores?", "Los polinizadores transportan polen entre flores y facilitan la reproducción de numerosas plantas, contribuyendo a la diversidad vegetal."),
    ("¿Qué consecuencias puede producir la desaparición de una especie?", "Puede alterar las cadenas alimentarias y otras relaciones ecológicas, especialmente cuando esa especie desempeña una función importante."),
    ("¿Qué significa conservar un hábitat?", "Significa proteger las condiciones ambientales y los recursos necesarios para que las especies puedan vivir y reproducirse."),
    ("¿Qué son los servicios ecosistémicos?", "Son los beneficios que las personas obtienen de los ecosistemas, como polinización, regulación del clima, provisión de alimentos y purificación del agua."),
    ("¿Cómo afecta la contaminación a los ecosistemas?", "La contaminación puede perjudicar organismos, modificar las condiciones del ambiente y alterar las interacciones ecológicas."),
    ("¿Qué es un cambio físico de la materia?", "Es una transformación que modifica propiedades físicas, como la forma o el estado, sin cambiar la identidad química de las sustancias."),
    ("¿Qué es un cambio químico?", "Es una transformación en la que se reorganizan los átomos mediante reacciones químicas y se originan sustancias diferentes."),
    ("¿Qué sucede cuando el hielo se derrite?", "El hielo pasa de sólido a líquido al absorber energía térmica. Las moléculas siguen siendo de agua, por lo que ocurre un cambio físico."),
    ("¿Qué ocurre durante la evaporación?", "Durante la evaporación, moléculas de la superficie de un líquido adquieren suficiente energía para pasar al estado gaseoso."),
    ("¿Qué ocurre durante una combustión?", "La combustión es una reacción química entre un combustible y un oxidante que libera energía y genera nuevas sustancias."),
    ("¿Qué indica la formación de sustancias nuevas?", "La aparición de sustancias con una composición química diferente es una característica fundamental de una transformación química."),
    ("¿Cómo influye la temperatura en los cambios de estado?", "Los cambios de temperatura modifican la energía de las partículas y pueden provocar transiciones entre los estados sólido, líquido y gaseoso."),
    ("¿Qué es la condensación?", "Es el cambio de estado gaseoso a líquido, que ocurre cuando el vapor pierde suficiente energía térmica bajo determinadas condiciones."),
    ("¿Qué es la digestión?", "Es el conjunto de procesos mecánicos y químicos que transforman los alimentos en componentes más pequeños para facilitar su absorción."),
    ("¿Dónde se absorbe la mayor parte de los nutrientes?", "La mayor parte de la absorción de nutrientes ocurre en el intestino delgado, cuya superficie permite su incorporación al organismo."),
    ("¿Qué son los nutrientes?", "Son sustancias presentes en los alimentos que el organismo utiliza para obtener energía, construir estructuras y regular procesos biológicos."),
    ("¿Qué es el metabolismo celular?", "Es el conjunto de reacciones químicas que ocurren en las células para transformar sustancias, obtener energía y mantener sus funciones."),
    ("¿Qué es la respiración celular?", "Es un conjunto de procesos metabólicos mediante los cuales las células obtienen energía utilizable a partir de moléculas orgánicas."),
    ("¿Qué es el ATP?", "El ATP es una molécula que participa en la transferencia de energía dentro de las células y permite impulsar numerosos procesos celulares."),
    ("¿Cómo se relacionan la glucosa y la producción de energía celular?", "La glucosa puede degradarse mediante procesos metabólicos que permiten producir ATP, una fuente inmediata de energía para muchas funciones celulares."),
    ("¿Qué función cumplen las mitocondrias?", "En las células eucariotas, las mitocondrias participan en la respiración celular aeróbica y en la producción de gran parte del ATP."),
    ("¿Qué es una célula?", "Es la unidad estructural y funcional básica de los seres vivos, capaz de realizar procesos necesarios para mantener la vida."),
    ("¿Qué función cumple la membrana plasmática?", "La membrana plasmática delimita la célula y regula el intercambio de sustancias con el ambiente."),
    ("¿Qué función cumple el núcleo celular?", "En las células eucariotas, el núcleo contiene la mayor parte del material genético y participa en la regulación de las actividades celulares."),
    ("¿Qué son los cloroplastos?", "Son orgánulos presentes en plantas y algas que contienen pigmentos fotosintéticos y permiten realizar la fotosíntesis."),
    ("¿Qué diferencia existe entre células animales y vegetales?", "Las células vegetales suelen tener pared celular, cloroplastos y una gran vacuola central. Las células animales carecen de pared celular y cloroplastos."),
    ("¿Qué es un organismo unicelular?", "Es un ser vivo formado por una sola célula, que realiza todas las funciones necesarias para su supervivencia."),
    ("¿Qué es un organismo pluricelular?", "Es un ser vivo formado por numerosas células que pueden especializarse y organizarse en tejidos y órganos."),
    ("¿Por qué necesitan energía las células?", "Las células requieren energía para transportar sustancias, sintetizar moléculas, mantener sus estructuras y realizar otras funciones vitales."),
    ("¿Qué es el ciclo del agua?", "Es el movimiento continuo del agua entre la superficie terrestre, la atmósfera y otros reservorios mediante procesos físicos."),
    ("¿Qué función cumple la evaporación en el ciclo del agua?", "La evaporación permite que el agua líquida pase a la atmósfera como vapor, impulsada por el intercambio de energía térmica."),
    ("¿Cómo se forman las nubes?", "Las nubes se forman cuando el vapor de agua del aire se enfría y se condensa sobre pequeñas partículas, produciendo gotas o cristales de hielo."),
    ("¿Qué es la precipitación?", "Es la caída de agua desde la atmósfera hacia la superficie terrestre en forma de lluvia, nieve, granizo u otras formas."),
    ("¿Qué diferencia existe entre tiempo atmosférico y clima?", "El tiempo atmosférico describe las condiciones de la atmósfera en un momento y lugar determinados. El clima describe sus patrones durante períodos prolongados."),
    ("¿Qué es el efecto invernadero natural?", "Es un proceso mediante el cual determinados gases atmosféricos absorben y emiten radiación infrarroja, contribuyendo a mantener la temperatura terrestre."),
    ("¿Cómo participan las plantas en el ciclo del carbono?", "Las plantas incorporan dióxido de carbono mediante la fotosíntesis y almacenan carbono en compuestos orgánicos. También liberan dióxido de carbono mediante la respiración."),
    ("¿Por qué es importante el agua para los seres vivos?", "El agua actúa como disolvente y medio para numerosas reacciones químicas, participa en el transporte de sustancias y ayuda a regular la temperatura.")
]

# Convierto los pares adicionales en un DataFrame.
dataframe_adicional = pd.DataFrame(
    preguntas_adicionales,
    columns=["pregunta", "respuesta"]
)

# Conservo las preguntas originales y agrego las nuevas.
dataframe_entrenamiento = pd.concat(
    [dataframe_entrenamiento_original, dataframe_adicional],
    ignore_index=True
).reset_index(drop=True)

# Muestro resultados.
print(f"Preguntas originales de entrenamiento: {len(dataframe_entrenamiento_original):,}")
print(f"Preguntas agregadas de entrenamiento: {len(dataframe_adicional):,}")
print(f"Total de preguntas de entrenamiento: {len(dataframe_entrenamiento):,}")

Preguntas originales de entrenamiento: 96
Preguntas agregadas de entrenamiento: 96
Total de preguntas de entrenamiento: 192


#### IV. Sumo preguntas para las pruebas.

In [32]:
# Creo manualmente el conjunto de pruebas.
dataframe_pruebas = pd.DataFrame([
    {"pregunta": "¿Por qué aparece una zona oscura detrás de un objeto que recibe luz?", "respuesta": "Porque el objeto bloquea la propagación de la luz e impide que llegue a la región situada detrás de él."},
    {"pregunta": "¿Cómo consiguen las plantas el agua y los minerales que necesitan para crecer?", "respuesta": "Los absorben del suelo mediante las raíces y los transportan hacia otras partes de la planta."},
    {"pregunta": "¿Por qué el cobre permite circular corriente eléctrica mientras que el plástico la dificulta?", "respuesta": "Porque el cobre posee electrones móviles que facilitan el flujo de cargas eléctricas, mientras que el plástico actúa como aislante."},
    {"pregunta": "¿Qué consecuencias puede tener la desaparición de los depredadores de una cadena alimentaria?", "respuesta": "Puede aumentar la población de presas y alterar el equilibrio de las relaciones alimentarias del ecosistema."},
    {"pregunta": "¿Por qué una vibración sonora puede desplazarse a distintas velocidades según el medio?", "respuesta": "Porque la velocidad del sonido depende de las propiedades elásticas y de la densidad del material por el que se propaga."},
    {"pregunta": "¿Qué mecanismos del organismo provocan que aumenten los latidos ante una situación de peligro?", "respuesta": "La activación del sistema nervioso simpático y la liberación de adrenalina aumentan la frecuencia cardíaca."},
    {"pregunta": "¿Por qué un objeto cae al suelo y un planeta puede mantenerse en órbita?", "respuesta": "Porque la atracción gravitatoria acelera los cuerpos y modifica sus trayectorias, provocando caídas y movimientos orbitales."},
    {"pregunta": "¿Cómo contribuye la variedad de especies a que un ecosistema resista perturbaciones?", "respuesta": "Las distintas especies cumplen funciones ecológicas que favorecen la estabilidad y la capacidad de recuperación del ecosistema."},
    {"pregunta": "¿Por qué derretir hielo y quemar madera representan transformaciones diferentes?", "respuesta": "Porque al derretir hielo solo cambia el estado físico del agua, mientras que al quemar madera ocurren reacciones químicas que generan sustancias nuevas."},
    {"pregunta": "¿Cómo aprovechan las células la energía contenida en los nutrientes después de la digestión?", "respuesta": "Mediante procesos metabólicos como la respiración celular, que permiten producir ATP para realizar las funciones celulares."}
])

# Muestro resultados.
print(f"Preguntas originales de prueba: 0")
print(f"Preguntas agregadas de prueba: {len(dataframe_pruebas):,}")
print(f"Total de preguntas de prueba: {len(dataframe_pruebas):,}")

Preguntas originales de prueba: 0
Preguntas agregadas de prueba: 10
Total de preguntas de prueba: 10


#### V. Tabla interactiva genérica (DataTable).

In [33]:
# Defino una tabla interactiva reutilizable.
def tabla_interactiva(dataframe, titulo):
    # Preparo una copia con identificadores únicamente para la visualización.
    dt_datos = dataframe.drop(columns=["id"], errors="ignore").copy()
    dt_datos.insert(0, "id", range(1, len(dt_datos) + 1))

    # Configuro las variables de búsqueda, ordenamiento y paginación.
    btn_anterior = widgets.Button(description="Anterior")
    btn_buscar = widgets.Button(description="Buscar")
    btn_siguiente = widgets.Button(description="Siguiente")
    dt_columna_orden = None
    dt_columnas = dt_datos.columns.tolist()
    dt_orden_ascendente = True
    dt_registros_por_pagina = 5
    dt_resultados = dt_datos
    dt_texto_busqueda = ""
    html_busqueda = widgets.Text(placeholder="Buscar en todo el dataset...", description="Buscar:", layout=widgets.Layout(width="300px"))
    html_informacion1 = widgets.HTML()
    html_informacion2 = widgets.HTML()
    html_pagina = widgets.BoundedIntText(value=1, min=1, max=1, description="Página:", layout=widgets.Layout(width="148px"))
    html_tabla = widgets.HTML()

    # Configuro los anchos de las columnas.
    dt_anchos = {"id": "6%", "pregunta": "44%", "respuesta": "50%"}

    btns_encabezados = [
        widgets.Button(
            description=columna,
            layout=widgets.Layout(width=dt_anchos.get(columna, "33%"))
        )
        for columna in dt_columnas
    ]

    def mostrar_pagina():
        total_paginas = max(1, (len(dt_resultados) + dt_registros_por_pagina - 1) // dt_registros_por_pagina)
        html_pagina.max = total_paginas

        inicio = (html_pagina.value - 1) * dt_registros_por_pagina
        muestra = dt_resultados.iloc[inicio:inicio + dt_registros_por_pagina].copy()

        html_informacion1.value = f" <b>Registros encontrados:</b> {len(dt_resultados):,}"
        html_informacion2.value = f" <b>Página ({dt_registros_por_pagina} registros):</b> {html_pagina.value:,} de {total_paginas:,}"

        btn_anterior.disabled = html_pagina.value <= 1
        btn_siguiente.disabled = html_pagina.value >= total_paginas

        # Preparo los textos sin modificar los datos originales.
        for columna in muestra.columns:
            muestra[columna] = muestra[columna].fillna("").astype(str)

        # Limito las respuestas únicamente en la visualización.
        muestra["respuesta"] = muestra["respuesta"].apply(
            lambda texto: texto[:50] + "..." if len(texto) > 50 else texto
        )

        html_tabla.value = (
            '<style>'
            '.tabla-datos { width: 100%; table-layout: fixed; border-collapse: collapse; }'
            '.tabla-datos td { padding: 10px 12px; text-align: left !important; '
            'vertical-align: middle; white-space: normal; overflow-wrap: break-word; '
            'border-bottom: 1px solid #ddd; }'
            '.tabla-datos td:nth-child(1) { width: 6%; text-align: center !important; }'
            '.tabla-datos td:nth-child(2) { width: 44%; }'
            '.tabla-datos td:nth-child(3) { width: 50%; }'
            '.tabla-datos tr:nth-child(odd) { background-color: #f5f5f5; }'
            '.tabla-datos tr:nth-child(even) { background-color: #fff; }'
            '</style>'
            '<div style="max-height: 800px; overflow: auto;">'
            + muestra.to_html(index=False, header=False, escape=True, classes="tabla-datos")
            + '</div>'
        )

    def actualizar_resultados():
        nonlocal dt_resultados

        dt_resultados = dt_datos

        if dt_texto_busqueda:
            filtro = dt_resultados.astype("string").apply(
                lambda columna: columna.str.contains(dt_texto_busqueda, case=False, na=False, regex=False)
            ).any(axis=1)
            dt_resultados = dt_resultados.loc[filtro]

        if dt_columna_orden is not None:
            dt_resultados = dt_resultados.sort_values(
                by=dt_columna_orden,
                ascending=dt_orden_ascendente,
                kind="stable",
                na_position="last"
            )

        html_pagina.value = 1
        mostrar_pagina()

    def buscar_registros(boton=None):
        nonlocal dt_texto_busqueda

        dt_texto_busqueda = html_busqueda.value.strip()
        actualizar_resultados()

    def ordenar_columna(columna):
        nonlocal dt_columna_orden, dt_orden_ascendente

        dt_orden_ascendente = not dt_orden_ascendente if dt_columna_orden == columna else True
        dt_columna_orden = columna

        for boton, nombre in zip(btns_encabezados, dt_columnas):
            simbolo = " ▲" if dt_orden_ascendente else " ▼"
            boton.description = nombre + simbolo if nombre == columna else nombre

        actualizar_resultados()

    def navegar(direccion):
        html_pagina.value = max(1, min(html_pagina.max, html_pagina.value + direccion))

    # Asocio los controles.
    for boton, columna in zip(btns_encabezados, dt_columnas):
        boton.on_click(lambda evento, nombre=columna: ordenar_columna(nombre))

    btn_buscar.on_click(buscar_registros)
    html_pagina.observe(lambda cambio: mostrar_pagina(), names="value")
    btn_anterior.on_click(lambda boton: navegar(-1))
    btn_siguiente.on_click(lambda boton: navegar(1))

    # Muestro la tabla.
    display(widgets.HTML(f"<h2>{titulo}</h2>"))
    display(widgets.HTML("<hr/>"))
    display(widgets.HBox([html_busqueda, btn_buscar, html_informacion1]))
    display(widgets.HTML("<hr/>"))
    display(widgets.HBox([html_pagina, btn_anterior, btn_siguiente, html_informacion2]))
    display(widgets.HTML("<hr/>"))
    display(widgets.HBox(btns_encabezados))
    display(html_tabla)

    mostrar_pagina()

#### VI. Muestro el conjunto de entrenamiento.

In [34]:
# Muestro el dataset de entrenamiento.
tabla_interactiva(dataframe_entrenamiento, "Tabla interactiva del conjunto de entrenamiento:")

HTML(value='<h2>Tabla interactiva del conjunto de entrenamiento:</h2>')

HTML(value='<hr/>')

HTML(value='<hr/>')

HTML(value='<hr/>')

HTML(value='')

#### VII. Muestro el conjunto de pruebas.

In [35]:
# Muestro el dataset de pruebas.
tabla_interactiva(dataframe_pruebas, "Tabla interactiva del conjunto de pruebas:")

HTML(value='<h2>Tabla interactiva del conjunto de pruebas:</h2>')

HTML(value='<hr/>')

HTML(value='<hr/>')

HTML(value='<hr/>')

HTML(value='')

### b) Debe elegir un modelo LLM de HuggingFace y al menos dos modelos de embeddings.

Justifique su elección.

#### I. Selección de modelos de Hugging Face.

In [36]:
# Defino el modelo LLM.
modelo_llm = "Qwen/Qwen2.5-1.5B-Instruct"

# Defino los modelos de embeddings.
modelo_embedding_1 = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
modelo_embedding_2 = "intfloat/multilingual-e5-small"

# Cargo el modelo LLM.
tokenizador_llm = AutoTokenizer.from_pretrained(modelo_llm)
modelo_llm_cargado = AutoModelForCausalLM.from_pretrained(
    modelo_llm,
    torch_dtype="auto",
    device_map="auto"
)
print(f"Modelo LLM cargado correctamente: {modelo_llm}")
print()

# Cargo el primer modelo de embeddings.
embedding_1_cargado = SentenceTransformer(modelo_embedding_1)
print(f"Modelo de embeddings 1 cargado correctamente: {modelo_embedding_1}")
print()

# Cargo el segundo modelo de embeddings.
embedding_2_cargado = SentenceTransformer(modelo_embedding_2)
print(f"Modelo de embeddings 2 cargado correctamente: {modelo_embedding_2}")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Modelo LLM cargado correctamente: Qwen/Qwen2.5-1.5B-Instruct



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modelo de embeddings 1 cargado correctamente: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modelo de embeddings 2 cargado correctamente: intfloat/multilingual-e5-small


#### II. Justificación de los modelos seleccionados.

> Para seleccionar los modelos, consideré principalmente su compatibilidad con el español, los recursos disponibles en Google Colab y su aplicación a un sistema de generación aumentada por recuperación (RAG).
>
> También prioricé modelos que permitieran trabajar con un conjunto de entrenamiento de 192 registros y comparar diferentes estrategias de recuperación semántica, utilizando las mismas 10 preguntas de prueba.

---

> **1. Modelo LLM: Qwen/Qwen2.5-1.5B-Instruct**
>
> Elegí este modelo por las siguientes características:
>
> - **Compatibilidad con español:** es un modelo multilingüe que permite interpretar preguntas y generar respuestas en español.
> - **Tamaño relativamente reducido:** tiene aproximadamente 1.500 millones de parámetros, por lo que requiere menos memoria y procesamiento que modelos de 7.000 u 8.000 millones.
> - **Versión Instruct:** está ajustado para seguir instrucciones, lo que permite indicarle que responda utilizando los contextos recuperados, sin incorporar información que no esté respaldada por ellos.
> - **Integración con Hugging Face:** puede cargarse mediante la biblioteca `transformers` y ejecutarse directamente en Google Colab, sin depender de una API externa.
>
> **Otras alternativas consideradas:**
>
> | Modelo | Parámetros aproximados | Motivo de elección o descarte |
> |---|---|---|
> | Qwen2.5-0.5B-Instruct | 500 millones | Probado inicialmente. Se reemplazó por errores e inconsistencias en las respuestas. |
> | **Qwen2.5-1.5B-Instruct** | **1.500 millones** | **Seleccionado por su equilibrio entre capacidad de generación y recursos necesarios.** |
> | SmolLM2-1.7B-Instruct | 1.700 millones | No priorizado por su enfoque principalmente en inglés. |
> | Gemma 3 1B | 1.000 millones | Alternativa compacta, pero menos adecuada para priorizar el español. |
> | Qwen2.5-3B-Instruct | 3.000 millones | No seleccionado por su mayor consumo de memoria y procesamiento. |
> | Qwen2.5-7B-Instruct | 7.000 millones | No seleccionado por sus mayores exigencias de recursos en Google Colab. |
>
> Inicialmente utilicé Qwen2.5-0.5B-Instruct, pero observé respuestas incompletas y errores científicos. Por este motivo, decidí utilizar Qwen2.5-1.5B-Instruct para mejorar la capacidad de generación sin aumentar excesivamente los recursos necesarios.
>
> En el chatbot, Qwen no realiza la búsqueda de información. Su función es recibir los contextos recuperados mediante los embeddings y FAISS, y generar una respuesta final en español.

---

> **2. Modelos de embeddings**
>
> Los modelos de embeddings transforman las preguntas en vectores numéricos que representan su significado. Estos vectores permiten recuperar información mediante similitud semántica, incluso cuando las consultas utilizan palabras diferentes a las preguntas almacenadas.
>
> **2.1. sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2**
>
> Elegí este modelo por las siguientes características:
>
> - **Soporte multilingüe:** permite representar textos en español, que es el idioma utilizado en el conjunto de entrenamiento.
> - **Tamaño reducido:** requiere relativamente pocos recursos, lo que facilita su ejecución en Google Colab.
> - **Embeddings de 384 dimensiones:** convierte cada pregunta en un vector de 384 valores numéricos. Esta cantidad está definida por la arquitectura del modelo y ofrece una representación relativamente compacta.
> - **Similitud semántica:** está diseñado para producir representaciones útiles al comparar oraciones y paráfrasis.
> - **Integración sencilla:** puede utilizarse mediante la biblioteca `sentence-transformers`, sin necesidad de entrenarlo desde cero.
>
> **Motivo de selección:** lo utilizaré como primer modelo de referencia para identificar preguntas similares dentro del conjunto de entrenamiento y comparar sus resultados con E5-small.
>
> **Limitación:** encontrar preguntas semánticamente similares no garantiza que sus respuestas contengan toda la información necesaria para responder correctamente una consulta.

---

> **2.2. intfloat/multilingual-e5-small**
>
> Elegí este modelo por las siguientes características:
>
> - **Soporte multilingüe:** permite procesar consultas y documentos en español.
> - **Tamaño reducido:** tiene aproximadamente 118 millones de parámetros, lo que facilita su ejecución en Google Colab.
> - **Embeddings de 384 dimensiones:** utiliza la misma cantidad de dimensiones que MiniLM, permitiendo comparar ambos modelos con representaciones de igual tamaño.
> - **Especialización en recuperación de información:** fue entrenado para relacionar consultas con textos relevantes, lo que resulta apropiado para un sistema RAG.
> - **Uso de prefijos:** utiliza `query:` para identificar las consultas y `passage:` para los textos almacenados, siguiendo el formato recomendado para este modelo.
> - **Integración con Hugging Face y FAISS:** permite generar embeddings mediante `sentence-transformers` y almacenarlos en índices vectoriales de FAISS.
>
> **Motivo de selección:** lo utilizaré como segundo modelo para comprobar si su enfoque orientado a recuperación de información permite encontrar contextos más relevantes que MiniLM.
>
> **Limitación:** aunque está especializado en recuperación semántica, no garantiza que los contextos encontrados sean suficientes o correctos para responder todas las preguntas.

---

> **3. Comparación de los modelos de embeddings**
>
> Seleccioné MiniLM y E5-small porque ambos son multilingües, relativamente livianos y generan embeddings de 384 dimensiones, pero presentan diferencias en sus enfoques de entrenamiento.
>
> | Característica | MiniLM | E5-small |
> |---|---|---|
> | Idioma español | Sí | Sí |
> | Dimensiones | 384 | 384 |
> | Tamaño | Reducido | Reducido |
> | Enfoque principal | Similitud entre oraciones | Recuperación de información |
> | Prefijos específicos | No requiere | `query:` y `passage:` |
> | Motor vectorial | FAISS | FAISS |
>
> Para realizar la comparación, utilizaré:
>
> - **El mismo LLM:** Qwen2.5-1.5B-Instruct.
> - **El mismo conjunto de entrenamiento:** 192 registros.
> - **El mismo conjunto de pruebas:** 10 preguntas con sus respuestas esperadas.
> - **El mismo motor vectorial:** FAISS.
> - **La misma cantidad de contextos recuperados:** los dos más similares a cada consulta.
> - **La misma configuración de generación:** instrucciones y parámetros de Qwen iguales para ambos modelos.
>
> Cada modelo generará sus propios embeddings y tendrá un índice FAISS independiente. De esta manera, las diferencias observadas estarán relacionadas principalmente con la recuperación semántica de cada modelo de embeddings.
>
> **El objetivo será determinar cuál de los dos modelos recupera información más relevante y analizar cómo esas diferencias influyen en la calidad de las respuestas generadas por Qwen.**

### c) Implemente una clase ChatBot usando lo elegido en b).

Puede usar cualquier base de datos vectorial: Chroma y FAISS son las más documentadas. Recuerde que sus datos para su BD conocimiento es el dataset que Ud. planteó en el TP3.

#### I. Genero embeddings.

In [37]:
# Preparo las preguntas del conjunto de entrenamiento.
textos_entrenamiento = dataframe_entrenamiento["pregunta"].fillna("").astype(str).tolist()

# Genero los embeddings con MiniLM.
embeddings_1 = embedding_1_cargado.encode(
    textos_entrenamiento,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(f"Embeddings MiniLM generados correctamente: {embeddings_1.shape}")
print()

# Preparo las preguntas para E5-small.
textos_entrenamiento_e5 = [f"passage: {texto}" for texto in textos_entrenamiento]

# Genero los embeddings con E5-small.
embeddings_2 = embedding_2_cargado.encode(
    textos_entrenamiento_e5,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(f"Embeddings E5-small generados correctamente: {embeddings_2.shape}")
print()

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embeddings MiniLM generados correctamente: (192, 384)



Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embeddings E5-small generados correctamente: (192, 384)



#### II. Creo la clase ChatBot.

In [39]:
# Creo la clase ChatBot para utilizar FAISS.
class ChatBot:
    def __init__(self, modelo_embedding, embeddings, prefijo=""):
        self.modelo_embedding = modelo_embedding
        self.prefijo = prefijo
        self.datos = dataframe_entrenamiento.reset_index(drop=True).copy()

        # Preparo los embeddings para FAISS.
        vectores = embeddings.astype("float32")

        # Creo un índice vectorial vacío con FAISS,
        # preparado para trabajar con 384 dimensiones.
        self.indice = faiss.IndexFlatIP(vectores.shape[1])

        # Cargo los 192 embeddings en ese índice.
        self.indice.add(vectores)

    def recuperar_contextos(self, pregunta, cantidad=2):
        # Genero el embedding de la consulta.
        consulta = self.modelo_embedding.encode(
            [self.prefijo + pregunta],
            convert_to_numpy=True,
            normalize_embeddings=True
        ).astype("float32")

        # Busco los registros más similares con FAISS.
        similitudes, posiciones = self.indice.search(
            consulta,
            min(cantidad, self.indice.ntotal)
        )

        # Recupero las preguntas y respuestas correspondientes.
        contextos = []

        for posicion, similitud in zip(posiciones[0], similitudes[0]):
            registro = self.datos.iloc[int(posicion)]

            contextos.append({
                "pregunta": str(registro["pregunta"]),
                "respuesta": str(registro["respuesta"]),
                "similitud": float(similitud)
            })

        return contextos

    def recuperar(self, pregunta):
        # Recupero el contexto más relevante para la evaluación.
        contexto = self.recuperar_contextos(pregunta, cantidad=1)[0]

        resultado = pd.Series({
            "pregunta": contexto["pregunta"],
            "respuesta": contexto["respuesta"]
        })

        return resultado, contexto["similitud"]

    def responder(self, pregunta, contextos=None):
        # Recupero los dos contextos más relevantes.
        if contextos is None:
            contextos = self.recuperar_contextos(pregunta, cantidad=2)

        # Preparo la información recuperada para Qwen.
        referencias = []

        for numero, contexto in enumerate(contextos, start=1):
            referencias.append(
                f"Fuente {numero}: {contexto['pregunta']} "
                f"Respuesta: {contexto['respuesta']}"
            )

        texto_referencias = " ".join(referencias)

        # Defino las instrucciones del modelo generativo.
        instrucciones = (
            "Sos un asistente educativo de ciencias naturales. "
            "Respondé en español con una o dos oraciones claras y completas. "
            "Utilizá solamente la información relevante de las fuentes. "
            "No inventes información ni repitas la pregunta. "
            "Si no hay información suficiente, respondé: "
            "No tengo información suficiente."
        )

        # Preparo los mensajes para Qwen.
        mensajes = [
            {"role": "system", "content": instrucciones},
            {
                "role": "user",
                "content": (
                    f"Fuentes: {texto_referencias} "
                    f"Pregunta: {pregunta}"
                )
            }
        ]

        # Tokenizo los mensajes.
        entradas = tokenizador_llm.apply_chat_template(
            mensajes,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True
        ).to(modelo_llm_cargado.device)

        # Genero la respuesta con Qwen.
        modelo_llm_cargado.eval()

        with torch.inference_mode():
            salida = modelo_llm_cargado.generate(
                **entradas,
                max_new_tokens=160,
                do_sample=False,
                pad_token_id=tokenizador_llm.eos_token_id
            )

        # Convierto los tokens generados en texto.
        respuesta = tokenizador_llm.decode(
            salida[0][entradas["input_ids"].shape[1]:],
            skip_special_tokens=True
        ).strip()

        return respuesta

print("Clase ChatBot definida correctamente.")
print()

# Creo los dos chatbots utilizando los embeddings existentes.
chatbot_minilm_faiss = ChatBot(
    embedding_1_cargado,
    embeddings_1
)

print("Chatbot MiniLM + FAISS creado correctamente.")

chatbot_e5_faiss = ChatBot(
    embedding_2_cargado,
    embeddings_2,
    prefijo="query: "
)

print("Chatbot E5-small + FAISS creado correctamente.")

Clase ChatBot definida correctamente.

Chatbot MiniLM + FAISS creado correctamente.
Chatbot E5-small + FAISS creado correctamente.


### d) Pruebe el chatbot creado en c) con las preguntas de su dataset a)

Usando los modelos elegidos en b), observe las respuestas generadas por el chatbot comparando al menos dos modelos de embeddings.

Justifique y determine cuál elegiría para su aplicación.

#### I. Preparo los chatbots.

In [42]:
# Utilizo todas las preguntas del conjunto de pruebas.
muestra = dataframe_pruebas.reset_index(drop=True)
total_preguntas = len(muestra)

# Defino las dos combinaciones para comparar.
chatbots = [
    ("MiniLM + FAISS", chatbot_minilm_faiss, "respuesta_minilm_faiss"),
    ("E5-small + FAISS", chatbot_e5_faiss, "respuesta_e5_faiss")
]

print(f"Preguntas de prueba: {total_preguntas}")
print(f"Combinaciones: {len(chatbots)}")

Preguntas de prueba: 10
Combinaciones: 2


#### II. Recupero los contextos.

In [43]:
# Recupero los contextos utilizando la clase ChatBot.
contextos_recuperados = {}

for _, fila in muestra.iterrows():
    pregunta = fila["pregunta"]
    contextos_recuperados[pregunta] = {}

    for nombre, chatbot, columna in chatbots:
        contextos_recuperados[pregunta][columna] = chatbot.recuperar_contextos(
            pregunta,
            cantidad=2
        )

print("Contextos recuperados correctamente.")
print(f"Preguntas procesadas: {len(contextos_recuperados)}")

Contextos recuperados correctamente.
Preguntas procesadas: 10


#### III. Genero y comparo las respuestas con Qwen.

**Nota sobre el rendimiento:** Se recomienda utilizar una GPU T4 en Google Colab para acelerar la generación de respuestas. El código también funciona con CPU, aunque el tiempo de ejecución es considerablemente mayor.

In [45]:
# Genero las respuestas utilizando la clase ChatBot.
resultados = []

for indice, fila in muestra.iterrows():
    pregunta = fila["pregunta"]

    print(f"Pregunta {indice + 1}/{total_preguntas}: {pregunta}", flush=True)

    respuestas = {}

    for nombre, chatbot, columna in chatbots:
        print(f"  {nombre}: generando respuesta...", flush=True)

        # Utilizo la recuperación realizada previamente.
        contextos = contextos_recuperados[pregunta][columna]

        # Genero la respuesta mediante el método de la clase.
        respuesta = chatbot.responder(
            pregunta,
            contextos=contextos
        )

        respuestas[columna] = respuesta

        print(f"  {nombre}: completado.", flush=True)

    # Conservo las respuestas para la comparación.
    resultados.append({
        "pregunta": pregunta,
        "respuesta_esperada": fila["respuesta"],
        **respuestas
    })

    print()

# Construyo el DataFrame de resultados.
dataframe_resultados = pd.DataFrame(resultados)

Pregunta 1/10: ¿Por qué aparece una zona oscura detrás de un objeto que recibe luz?
  MiniLM + FAISS: generando respuesta...
  MiniLM + FAISS: completado.
  E5-small + FAISS: generando respuesta...
  E5-small + FAISS: completado.

Pregunta 2/10: ¿Cómo consiguen las plantas el agua y los minerales que necesitan para crecer?
  MiniLM + FAISS: generando respuesta...
  MiniLM + FAISS: completado.
  E5-small + FAISS: generando respuesta...
  E5-small + FAISS: completado.

Pregunta 3/10: ¿Por qué el cobre permite circular corriente eléctrica mientras que el plástico la dificulta?
  MiniLM + FAISS: generando respuesta...
  MiniLM + FAISS: completado.
  E5-small + FAISS: generando respuesta...
  E5-small + FAISS: completado.

Pregunta 4/10: ¿Qué consecuencias puede tener la desaparición de los depredadores de una cadena alimentaria?
  MiniLM + FAISS: generando respuesta...
  MiniLM + FAISS: completado.
  E5-small + FAISS: generando respuesta...
  E5-small + FAISS: completado.

Pregunta 5/10: ¿

#### IV. Muestro las respuestas una debajo de otra.

In [46]:
# Muestro las respuestas con formato compacto y títulos destacados.
contenido = """
  <h3>RESULTADOS DE LA COMPARACIÓN</h3>
  """

for indice, fila in dataframe_resultados.iterrows():
    contenido += f"""
    <div style="font-size: 13px; line-height: 2.0;">
        <hr style="margin: 10px 0; opacity: 0.25;">
        <div><b>PREGUNTA {indice + 1}/{len(dataframe_resultados)}:</b> {escape(str(fila['pregunta']))}</div>
        <div><b>Respuesta esperada:</b> {escape(str(fila['respuesta_esperada']))}</div>
        <div><b>Respuesta MiniLM + FAISS:</b> {escape(str(fila['respuesta_minilm_faiss']))}</div>
        <div><b>Respuesta E5-small + FAISS:</b> {escape(str(fila['respuesta_e5_faiss']))}</div>
    </div>
    """

display(HTML(contenido))

#### V. Conclusiones de la comparación.

> Al comparar las respuestas generadas con MiniLM y E5-small, observé que ambos modelos lograron responder correctamente varias preguntas del conjunto de prueba, aunque también presentaron respuestas incompletas y algunos errores científicos.
>
> MiniLM produjo respuestas más completas en determinados casos, como la pregunta sobre la desaparición de depredadores. E5-small, en cambio, identificó información más relevante en la consulta sobre la adrenalina y la respuesta del organismo ante situaciones de peligro.
>
> También observé que ambos sistemas generaron algunas respuestas idénticas y compartieron errores, como la definición incorrecta de ATP. Esto demuestra que recuperar información semánticamente relacionada no garantiza que el modelo generativo la utilice correctamente.
>
> **Por lo tanto, no identifiqué un ganador definitivo únicamente a partir de la calidad de las respuestas.** Para complementar esta comparación, evaluaré la relevancia y la cobertura de los contextos recuperados mediante las métricas del BONUS e).

### e) BONUS (opcional)

Evalue el chatbot creado en c) para los modelos de embeddings elegidos y usados en d) usando el dataset a) con las métricas context precision y context recall. Puede probar usar la librería ragas o implementar Ud. mismo el cálculo de dichas métricas.

Dado sus resultados: ¿refuerza o no sus conclusiones realizadas en d) ?

Explique los resultados obtenidos y agregue sus conclusiones.

#### I. Evalúo la recuperación de contexto de los dos chatbots.

In [50]:
# Utilizo ambos modelos para calcular la similitud semántica.
modelos_evaluacion = [
    ("MiniLM", embedding_1_cargado),
    ("E5-small", embedding_2_cargado)
]

def dividir_fragmentos(texto):
    # Divido las respuestas en fragmentos para compararlos.
    fragmentos = re.split(r"[.;:!?]+|,\s*", str(texto))
    return [fragmento.strip() for fragmento in fragmentos if fragmento.strip()]


def calcular_similitudes(fragmentos_contexto, fragmentos_esperados):
    # Comparo los fragmentos utilizando ambos modelos de embeddings.
    similitudes = []

    for nombre, modelo in modelos_evaluacion:
        textos = fragmentos_contexto + fragmentos_esperados

        # E5-small necesita identificar los textos como passages.
        if nombre == "E5-small":
            textos = [f"passage: {texto}" for texto in textos]

        vectores = modelo.encode(
            textos,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False
        )

        cantidad = len(fragmentos_contexto)
        matriz = vectores[:cantidad] @ vectores[cantidad:].T
        similitudes.append(matriz)

    # Promedio las similitudes de MiniLM y E5-small.
    return np.clip(np.mean(similitudes, axis=0), 0, 1)


# Evalúo los contextos previamente recuperados, sin ejecutar Qwen.
resultados_contexto = []

for _, fila in muestra.iterrows():
    pregunta = fila["pregunta"]
    respuesta_esperada = fila["respuesta"]
    fragmentos_esperados = dividir_fragmentos(respuesta_esperada)

    for nombre, chatbot, columna in chatbots:
        contextos = contextos_recuperados[pregunta][columna]

        # Uno las dos respuestas recuperadas.
        contexto = " ".join(
            registro["respuesta"] for registro in contextos
        )

        fragmentos_contexto = dividir_fragmentos(contexto)

        # Calculo las similitudes entre el contexto y la respuesta esperada.
        matriz = calcular_similitudes(
            fragmentos_contexto,
            fragmentos_esperados
        )

        # Precision: relevancia semántica de la información recuperada.
        context_precision = float(np.mean(np.max(matriz, axis=1)))

        # Recall: cobertura semántica de la respuesta esperada.
        context_recall = float(np.mean(np.max(matriz, axis=0)))

        resultados_contexto.append({
            "combinacion": nombre,
            "pregunta": pregunta,
            "contexto_recuperado": contexto,
            "context_precision": context_precision,
            "context_recall": context_recall
        })


# Guardo los resultados individuales.
dataframe_contexto = pd.DataFrame(resultados_contexto)

# Calculo los promedios de cada chatbot.
resumen_contexto = (
    dataframe_contexto
    .groupby("combinacion", sort=False)[
        ["context_precision", "context_recall"]
    ]
    .mean()
    .reset_index()
)

print("BONUS: EVALUACIÓN SEMÁNTICA DE RECUPERACIÓN")
print()
print("Context Precision: relevancia semántica del contexto recuperado.")
print("Context Recall: cobertura semántica de la respuesta esperada.")
print("Métricas aproximadas mediante similitud de embeddings.")
print()
print("RESULTADOS PROMEDIO:")

display(resumen_contexto.round(3))

BONUS: EVALUACIÓN SEMÁNTICA DE RECUPERACIÓN

Context Precision: relevancia semántica del contexto recuperado.
Context Recall: cobertura semántica de la respuesta esperada.
Métricas aproximadas mediante similitud de embeddings.

RESULTADOS PROMEDIO:


,combinacion,context_precision,context_recall
0,MiniLM + FAISS,0.708,0.782
1,E5-small + FAISS,0.716,0.839


#### II. Conclusiones de la evaluación de recuperación.

> Los resultados muestran que **E5-small + FAISS obtuvo un mejor rendimiento de recuperación semántica** que MiniLM + FAISS.
>
> En Context Precision, E5-small alcanzó 0,716 frente a 0,708 de MiniLM, mientras que en Context Recall obtuvo 0,839 frente a 0,782. La diferencia más significativa se observó en Context Recall, lo que indica que E5-small logró recuperar información con mayor cobertura respecto de las respuestas esperadas.
>
> Estos resultados complementan la comparación realizada en el apartado d), donde ambos modelos generaron respuestas correctas, parciales y algunas incorrectas. Aunque E5-small obtuvo mejores métricas de recuperación, esto no garantizó una mejor respuesta final en todos los casos, ya que Qwen también puede introducir errores durante la generación.
>
> **Concluyo que E5-small + FAISS es la combinación más conveniente para la recuperación de información en este trabajo**, aunque la calidad de las respuestas generadas todavía presenta limitaciones.
>
> Las métricas utilizadas son aproximaciones basadas en similitud semántica, por lo que no equivalen a una evaluación completa de Context Precision y Context Recall mediante RAGAS.

### f) BONUS (opcional)

Evalue el modelo LLM usado para el chatbot creado en c) con el modelo de embedding que mejor le haya resultado en d) usando el dataset a).

Use las métricas Answer Relevancy  y Faithfulness. Puede probar usar la librería ragas o implementar Ud. mismo el cálculo de dichas métricas. Explique los resultados obtenidos y agregue sus conclusiones.

#### I. Evalúo Answer Relevancy y Faithfulness con el mejor modelo de embeddings.

In [51]:
# Selecciono el modelo con mejor recuperación en el BONUS e).
mejor_modelo = resumen_contexto.copy()

mejor_modelo["promedio"] = mejor_modelo[
    ["context_precision", "context_recall"]
].mean(axis=1)

mejor_combinacion = mejor_modelo.sort_values(
    "promedio",
    ascending=False
).iloc[0]["combinacion"]

# Identifico la columna de respuestas del modelo seleccionado.
columna_respuesta = next(
    columna for nombre, chatbot, columna in chatbots
    if nombre == mejor_combinacion
)

# Recupero los contextos correspondientes.
contextos_evaluacion = (
    dataframe_contexto[
        dataframe_contexto["combinacion"] == mejor_combinacion
    ]
    .set_index("pregunta")["contexto_recuperado"]
    .to_dict()
)

# Evalúo las respuestas generadas sin volver a ejecutar Qwen.
resultados_llm = []

for _, fila in dataframe_resultados.iterrows():
    pregunta = fila["pregunta"]
    respuesta_esperada = str(fila["respuesta_esperada"])
    respuesta_generada = str(fila[columna_respuesta])
    contexto = contextos_evaluacion[pregunta]

    # Calculo Answer Relevancy comparando la respuesta
    # con la pregunta y la respuesta esperada.
    similitudes = calcular_similitudes(
        [respuesta_generada],
        [pregunta, respuesta_esperada]
    )

    answer_relevancy = float(np.mean(similitudes))

    # Calculo Faithfulness comparando los fragmentos
    # generados con la información recuperada.
    fragmentos_respuesta = dividir_fragmentos(respuesta_generada)
    fragmentos_contexto = dividir_fragmentos(contexto)

    if fragmentos_respuesta and fragmentos_contexto:
        matriz = calcular_similitudes(
            fragmentos_respuesta,
            fragmentos_contexto
        )

        faithfulness = float(np.mean(np.max(matriz, axis=1)))
    else:
        faithfulness = 0.0

    # Guardo los resultados de cada pregunta.
    resultados_llm.append({
        "pregunta": pregunta,
        "respuesta_generada": respuesta_generada,
        "answer_relevancy": answer_relevancy,
        "faithfulness": faithfulness
    })

# Construyo el DataFrame con los resultados.
dataframe_evaluacion_llm = pd.DataFrame(resultados_llm)

# Calculo los promedios de las métricas.
resumen_llm = (
    dataframe_evaluacion_llm[
        ["answer_relevancy", "faithfulness"]
    ]
    .mean()
    .to_frame(name="promedio")
    .reset_index()
    .rename(columns={"index": "metrica"})
)

print("BONUS: EVALUACIÓN SEMÁNTICA DEL MODELO LLM")
print()
print(f"Combinación evaluada: {mejor_combinacion}")
print(f"Modelo generativo: {modelo_llm}")
print("Métricas aproximadas mediante similitud de embeddings.")
print()
print("RESULTADOS POR PREGUNTA:")

display(
    dataframe_evaluacion_llm[
        ["pregunta", "answer_relevancy", "faithfulness"]
    ].round(3)
)

print()
print("RESULTADOS PROMEDIO:")

display(resumen_llm.round(3))

BONUS: EVALUACIÓN SEMÁNTICA DEL MODELO LLM

Combinación evaluada: E5-small + FAISS
Modelo generativo: Qwen/Qwen2.5-1.5B-Instruct
Métricas aproximadas mediante similitud de embeddings.

RESULTADOS POR PREGUNTA:


,pregunta,answer_relevancy,faithfulness
0,¿Por qué aparece una zona oscura detrás de un ...,0.885,0.891
1,¿Cómo consiguen las plantas el agua y los mine...,0.814,0.829
2,¿Por qué el cobre permite circular corriente e...,0.892,0.720
3,¿Qué consecuencias puede tener la desaparición...,0.865,0.894
4,¿Por qué una vibración sonora puede desplazars...,0.858,0.848
5,¿Qué mecanismos del organismo provocan que aum...,0.767,0.976
6,¿Por qué un objeto cae al suelo y un planeta p...,0.858,0.911
7,¿Cómo contribuye la variedad de especies a que...,0.864,0.737
8,¿Por qué derretir hielo y quemar madera repres...,0.872,0.774
9,¿Cómo aprovechan las células la energía conten...,0.877,0.811



RESULTADOS PROMEDIO:


,metrica,promedio
0,answer_relevancy,0.855
1,faithfulness,0.839


#### II. Conclusiones de la evaluación del modelo generativo.

> Para evaluar las respuestas de Qwen2.5-1.5B-Instruct, utilicé la combinación **E5-small + FAISS**, seleccionada por haber obtenido mejores resultados de recuperación en el BONUS e).
>
> Los resultados promedio fueron **0,855 en Answer Relevancy** y **0,839 en Faithfulness**. Estos valores indican una alta similitud semántica de las respuestas con las preguntas, las respuestas esperadas y los contextos recuperados.
>
> Sin embargo, la evaluación cualitativa permitió identificar errores que estas métricas no detectaron completamente. Por ejemplo, el modelo generó una definición incorrecta de ATP, a pesar de obtener valores relativamente elevados.
>
> **Concluyo que Qwen logró generar respuestas generalmente relacionadas con las preguntas y los contextos recuperados, aunque todavía presenta limitaciones de precisión científica.**
>
> Las métricas utilizadas son aproximaciones basadas en similitud semántica y no equivalen a las métricas originales de RAGAS. Por lo tanto, deben interpretarse junto con la revisión cualitativa de las respuestas.

### g) REFERENCIAS. Es obligatorio citar las fuentes de todo material utilizado para su TP incluido chats con IA generativa. Al final en su apartado Referencias liste todas las fuentes consultadas. Por ejemplo si mantuvo una conversación con ChatGPT puede agregar el enlace compartido a dicha conversación.

#### I. **Fuentes utilizadas:**

- Notebooks y materiales de clase.
- Búsquedas en Internet.
- Dataset original del TP3 y preguntas adicionales.
- Modelos de Hugging Face: Qwen, MiniLM y E5-small.
- Documentación de Python, Pandas y PyTorch.
- Documentación de FAISS, Transformers y Sentence Transformers.
- ChatGPT para resolver dudas, desarrollar ciertos fragmentos de código y analizar algunos resultados.

#### II. **Conclusión final del trabajo práctico.**

> El sistema presenta una recuperación semántica razonablemente buena, especialmente con **E5-small**, que obtuvo mejores resultados en Context Precision y Context Recall.
>
> FAISS permite recuperar eficientemente los contextos asociados a las preguntas más similares. Sin embargo, la principal limitación observada aparece durante la generación de respuestas con **Qwen2.5-1.5B-Instruct**.
>
> Aunque elegí un modelo relativamente liviano para ejecutarlo en Google Colab, esto implica un compromiso entre recursos computacionales y capacidad de generación.
>
> Los errores científicos observados muestran que una recuperación adecuada no garantiza necesariamente una respuesta correcta. Como mejora futura, evaluaría un modelo generativo de mayor capacidad, manteniendo la arquitectura de recuperación existente.

---

> **Evaluación general del sistema (escala de 1 a 10)**
>
> | Componente | Puntuación | Justificación |
> |---|---:|---|
> | Embeddings MiniLM | 7,8/10 | Buena recuperación semántica, aunque con menor cobertura que E5-small. |
> | Embeddings E5-small | 8,5/10 | Mejor rendimiento en recuperación y cobertura semántica. |
> | FAISS | 9,0/10 | Búsqueda vectorial exacta, eficiente y adecuada para los 192 registros. |
> | Generación Qwen 1.5B | 5,5/10 | Respuestas generalmente pertinentes, pero con errores científicos e información incompleta. |
> | **Sistema RAG completo** | **7,0/10** | **Arquitectura funcional, con una recuperación adecuada y limitaciones en la generación de respuestas.** |
>
> **Nota:** Las puntuaciones son valoraciones cualitativas orientativas basadas en los resultados obtenidos y la revisión de las respuestas. No representan porcentajes de precisión ni una medición formal de robustez.

---

> **Reflexión final**
>
> Este trabajo me permitió comprender e implementar las etapas principales de un sistema RAG: generación de embeddings, almacenamiento y búsqueda vectorial con FAISS, recuperación de contextos y generación de respuestas mediante un LLM.
>
> También pude comprobar que **la calidad de la recuperación y la calidad de la generación son aspectos diferentes**. Un modelo de embeddings puede identificar información relevante y, aun así, el modelo generativo puede interpretarla incorrectamente.
>
> El principal desafío fue encontrar un equilibrio entre la calidad de las respuestas y los recursos computacionales disponibles en Google Colab. Considero que el sistema cumple su objetivo como implementación educativa de RAG, aunque todavía requiere mejoras para ofrecer respuestas científicas más confiables.